In [7]:
import os
!git clone https://github.com/jiayili6-stack/telco-churn-ml.git
!cd telco-churn-ml/dashboard
!pip install -r requirements.txt
!streamlit run app.py

fatal: destination path 'telco-churn-ml' already exists and is not an empty directory.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
/bin/bash: line 1: streamlit: command not found


In [8]:
!pip install openpyxl

In [9]:
!pip install pymysql

In [10]:
import pandas as pd
import numpy as np
!pip install boto3
import boto3
from io import BytesIO

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

import joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 3.7 MB/s eta 0:00:00


In [11]:
import boto3

s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket='mlba-churnproject-rachelz3')

for obj in response.get('Contents', []):
    print(obj['Key'])

NoCredentialsError: Unable to locate credentials

In [ ]:
bucket_name = 'mlba-churnproject-rachelz3'
s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket=bucket_name)

file_key = None
for obj in response.get('Contents', []):
    if 'Telco' in obj['Key'] and obj['Key'].endswith('.xlsx'):
        file_key = obj['Key']

if file_key is None:
    raise Exception("❌ No Excel file found in S3")

print("✅ Using file:", file_key)

obj = s3.get_object(Bucket=bucket_name, Key=file_key)

df = pd.read_excel(BytesIO(obj['Body'].read()))

print("Columns BEFORE cleaning:")
print(df.columns)

In [ ]:
import pandas as pd
import boto3
from io import BytesIO

bucket_name = 'mlba-churnproject-rachelz3'
s3 = boto3.client('s3')

file_key = 'Telco-Customer-Churn.xlsx'  # ← use your confirmed key

obj = s3.get_object(Bucket=bucket_name, Key=file_key)

df = pd.read_excel(BytesIO(obj['Body'].read()))

# CLEAN COLUMN NAMES
df.columns = df.columns.str.strip()

print("Columns:", df.columns.tolist())

In [ ]:
customer_ids = df['customerID'].copy()

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# CREATE MODEL DATASET (separate)
df_model = df.drop('customerID', axis=1)

df_model['Churn'] = df_model['Churn'].map({'Yes':1, 'No':0})

In [ ]:
df_model = pd.get_dummies(df_model, drop_first=True)

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, customer_ids,
    test_size=0.25,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.fit(X_train_scaled, y_train)

In [ ]:
import numpy as np

y_prob = model.predict_proba(X_test_scaled)[:,1]
y_pred = (y_prob > 0.5).astype(int)

results = pd.DataFrame({
    'customerID': id_test,
    'churn_probability': y_prob,
    'churn_prediction': y_pred
})

print(results.head())

In [ ]:
import pymysql

# ================================
# CONNECT TO EXISTING DATABASE
# ================================
conn = pymysql.connect(
    host='churn-db.ctees2ym0jhl.us-east-1.rds.amazonaws.com',
    user='admin',
    password='Zhangruichen0118',
    database='churn_db'   # ✅ it already exists
)

cursor = conn.cursor()

print("✅ Connected to churn_db successfully")

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS predictions (
    id INT AUTO_INCREMENT PRIMARY KEY,
    customerID VARCHAR(50),
    churn_probability FLOAT,
    churn_prediction INT,
    timestamp DATETIME
)
""")

print("✅ Table ready")

In [ ]:
from datetime import datetime

for i in range(len(results)):
    cursor.execute("""
        INSERT INTO predictions (customerID, churn_probability, churn_prediction, timestamp)
        VALUES (%s, %s, %s, %s)
    """, (
        str(results.iloc[i]['customerID']),
        float(results.iloc[i]['churn_probability']),
        int(results.iloc[i]['churn_prediction']),
        datetime.now()
    ))

conn.commit()
conn.close()

print("✅ Data inserted into RDS")